In [1]:
import pandas as pd
import numpy as np

matches_df = pd.read_csv('../data/processed/matches.csv')

league_trends = matches_df.groupby(['league', 'year']).agg(
    matches=('match_id', 'count'),
    avg_goals=('total_goals', 'mean'),
    avg_xG=('total_xG', 'mean')
).round(3)

print(league_trends)

                     matches  avg_goals  avg_xG
league         year                            
bundesliga     2020      306      3.033   2.860
               2021      306      3.118   3.120
               2022      306      3.173   3.027
               2023      306      3.219   3.336
               2024      306      3.134   3.261
la_liga        2020      380      2.508   2.504
               2021      380      2.503   2.631
               2022      380      2.513   2.774
               2023      380      2.645   2.875
               2024      380      2.618   2.850
ligue_1        2020      380      2.761   2.658
               2021      380      2.808   2.718
               2022      380      2.808   2.944
               2023      306      2.699   2.964
               2024      306      2.977   3.297
premier_league 2020      380      2.695   2.735
               2021      380      2.818   2.839
               2022      380      2.853   2.965
               2023      380      3.279 

In [2]:
shots_df = pd.read_csv("/Users/aadvikmazumdar/Projects/OnTarget/data/processed/shots_enriched.csv")
shots_by_league_season = shots_df.groupby(['league', 'year']).size().reset_index(name='total_shots')

league_full = matches_df.groupby(['league', 'year']).agg(
    matches=('match_id', 'count'),
    total_goals=('total_goals', 'sum'),
    total_xG=('total_xG', 'sum')
).reset_index()

league_full = league_full.merge(shots_by_league_season, on=['league', 'year'])

league_full['avg_shots_per_match'] = (league_full['total_shots'] / league_full['matches']).round(2)
league_full['avg_goals_per_match'] = (league_full['total_goals'] / league_full['matches']).round(3)
league_full['avg_xG_per_match'] = (league_full['total_xG'] / league_full['matches']).round(3)
league_full['shot_conversion'] = (league_full['total_goals'] / league_full['total_shots']).round(4)

print(league_full[['league', 'year', 'avg_shots_per_match', 'avg_goals_per_match', 'avg_xG_per_match', 'shot_conversion']])

            league  year  avg_shots_per_match  avg_goals_per_match  \
0       bundesliga  2020                24.80                3.033   
1       bundesliga  2021                25.98                3.118   
2       bundesliga  2022                25.54                3.173   
3       bundesliga  2023                27.80                3.219   
4       bundesliga  2024                25.96                3.134   
5          la_liga  2020                21.40                2.508   
6          la_liga  2021                23.62                2.503   
7          la_liga  2022                24.67                2.513   
8          la_liga  2023                24.50                2.645   
9          la_liga  2024                23.83                2.618   
10         ligue_1  2020                23.48                2.761   
11         ligue_1  2021                24.03                2.808   
12         ligue_1  2022                24.59                2.808   
13         ligue_1  

In [3]:
pd.set_option('display.max_rows', None)
print(league_full[['league', 'year', 'avg_shots_per_match', 'avg_goals_per_match', 'avg_xG_per_match', 'shot_conversion']])

            league  year  avg_shots_per_match  avg_goals_per_match  \
0       bundesliga  2020                24.80                3.033   
1       bundesliga  2021                25.98                3.118   
2       bundesliga  2022                25.54                3.173   
3       bundesliga  2023                27.80                3.219   
4       bundesliga  2024                25.96                3.134   
5          la_liga  2020                21.40                2.508   
6          la_liga  2021                23.62                2.503   
7          la_liga  2022                24.67                2.513   
8          la_liga  2023                24.50                2.645   
9          la_liga  2024                23.83                2.618   
10         ligue_1  2020                23.48                2.761   
11         ligue_1  2021                24.03                2.808   
12         ligue_1  2022                24.59                2.808   
13         ligue_1  

In [4]:
def flag_season_anomalies(df, metric):
    stats = df.groupby('league')[metric].agg(['mean', 'std']).rename(columns={'mean': f'{metric}_mean', 'std': f'{metric}_std'})
    merged = df.merge(stats, on='league')
    merged[f'{metric}_zscore'] = (merged[metric] - merged[f'{metric}_mean']) / merged[f'{metric}_std']
    return merged[['league', 'year', metric, f'{metric}_zscore']]

shots_anomalies = flag_season_anomalies(league_full, 'avg_shots_per_match')
print(shots_anomalies[shots_anomalies['avg_shots_per_match_zscore'].abs() > 1.0].sort_values('avg_shots_per_match_zscore', ascending=False))

            league  year  avg_shots_per_match  avg_shots_per_match_zscore
3       bundesliga  2023                27.80                    1.613123
18  premier_league  2023                27.69                    1.525075
13         ligue_1  2023                25.58                    1.280254
21         serie_a  2021                26.14                    1.218016
0       bundesliga  2020                24.80                   -1.099528
15  premier_league  2020                24.30                   -1.233732
10         ligue_1  2020                23.48                   -1.290046
24         serie_a  2024                24.32                   -1.459265
5          la_liga  2020                21.40                   -1.684441


In [5]:
print(shots_df[shots_df['year'] == 2023].groupby('league')['match_id'].nunique())
print()
print(matches_df[matches_df['year'] == 2023].groupby('league')['match_id'].nunique())

league
bundesliga        306
la_liga           380
ligue_1           306
premier_league    380
serie_a           380
Name: match_id, dtype: int64

league
bundesliga        306
la_liga           380
ligue_1           306
premier_league    380
serie_a           380
Name: match_id, dtype: int64


In [6]:
team_season_shots = shots_df.groupby(['team_name', 'league', 'year']).agg(
    total_shots=('team_name', 'count'),
    total_goals=('result', lambda x: (x == 'Goal').sum()),
    avg_xG=('xG', 'mean')
).reset_index()

team_season_shots['conversion'] = (team_season_shots['total_goals'] / team_season_shots['total_shots']).round(4)

# z-score within each league-season (not across the whole dataset)
def zscore_within_group(df, metric, group_cols):
    stats = df.groupby(group_cols)[metric].agg(['mean', 'std'])
    merged = df.merge(stats, on=group_cols)
    merged[f'{metric}_zscore'] = (merged[metric] - merged['mean']) / merged['std']
    return merged.drop(columns=['mean', 'std'])

team_season_shots = zscore_within_group(team_season_shots, 'total_shots', ['league', 'year'])
team_season_shots = zscore_within_group(team_season_shots, 'conversion', ['league', 'year'])

print(team_season_shots.sort_values('total_shots_zscore', ascending=False).head(10))
print()
print(team_season_shots.sort_values('conversion_zscore', ascending=False).head(10))

               team_name          league  year  total_shots  total_goals  \
64         Bayern Munich      bundesliga  2024          646           96   
355          Real Madrid         la_liga  2021          657           80   
61         Bayern Munich      bundesliga  2021          673           92   
62         Bayern Munich      bundesliga  2022          630           90   
337  Paris Saint Germain         ligue_1  2024          638           90   
251            Liverpool  premier_league  2023          792           80   
258                 Lyon         ligue_1  2020          614           77   
311               Napoli         serie_a  2023          650           55   
54             Barcelona         la_liga  2024          677           99   
249            Liverpool  premier_league  2021          730           94   

       avg_xG  conversion  total_shots_zscore  conversion_zscore  
64   0.148132      0.1486            3.214533           1.372746  
355  0.124496      0.1218    

In [7]:
team_season_shots['combined_zscore'] = team_season_shots['total_shots_zscore'] + team_season_shots['conversion_zscore']
print(team_season_shots.sort_values('combined_zscore', ascending=False).head(10))

               team_name      league  year  total_shots  total_goals  \
64         Bayern Munich  bundesliga  2024          646           96   
54             Barcelona     la_liga  2024          677           99   
60         Bayern Munich  bundesliga  2020          576           98   
355          Real Madrid     la_liga  2021          657           80   
334  Paris Saint Germain     ligue_1  2021          564           88   
62         Bayern Munich  bundesliga  2022          630           90   
336  Paris Saint Germain     ligue_1  2023          513           78   
61         Bayern Munich  bundesliga  2021          673           92   
337  Paris Saint Germain     ligue_1  2024          638           90   
333  Paris Saint Germain     ligue_1  2020          569           85   

       avg_xG  conversion  total_shots_zscore  conversion_zscore  \
64   0.148132      0.1486            3.214533           1.372746   
54   0.148488      0.1462            2.484813           1.941551   
60 

In [8]:
anomaly_counts = team_season_shots[
    (team_season_shots['total_shots_zscore'] > 1.5) | (team_season_shots['conversion_zscore'] > 1.5)
].groupby('team_name').size().sort_values(ascending=False)

print(anomaly_counts.head(15))

team_name
Real Madrid               5
Paris Saint Germain       5
Bayern Munich             5
Inter                     5
Manchester City           4
Liverpool                 4
Napoli                    3
Barcelona                 3
Monaco                    2
Lyon                      2
RasenBallsport Leipzig    2
Lazio                     2
Fiorentina                2
Borussia Dortmund         2
Bayer Leverkusen          2
dtype: int64


In [9]:
efficient_low_volume = team_season_shots[
    (team_season_shots['total_shots_zscore'] < 0) & (team_season_shots['conversion_zscore'] > 1.0)
].sort_values('conversion_zscore', ascending=False)

print(efficient_low_volume)

                  team_name          league  year  total_shots  total_goals  \
219                   Lazio         serie_a  2021          455           74   
93                Brentford  premier_league  2024          444           64   
220                   Lazio         serie_a  2022          439           59   
115              Celta Vigo         la_liga  2020          359           55   
423               Tottenham  premier_league  2020          447           66   
341  RasenBallsport Leipzig      bundesliga  2021          438           72   
233               Leicester  premier_league  2021          435           62   
169              Fiorentina         serie_a  2024          448           58   
417              Strasbourg         ligue_1  2024          359           54   
297             Montpellier         ligue_1  2022          445           64   
380                    Roma         serie_a  2023          479           64   
451                  Verona         serie_a  2021   

In [10]:
team_total_goals = shots_df.groupby('team_name').agg(
    total_goals=('result', lambda x: (x == 'Goal').sum()),
    total_shots=('team_name', 'count')
).sort_values('total_goals', ascending=False)

print(team_total_goals.head(15))

                     total_goals  total_shots
team_name                                    
Bayern Munich                469         3163
Manchester City              435         3222
Paris Saint Germain          427         2854
Inter                        398         3031
Liverpool                    395         3388
Barcelona                    392         2947
Real Madrid                  379         3086
Borussia Dortmund            373         2508
Atalanta                     364         2837
Arsenal                      349         2845
Napoli                       343         2996
Bayer Leverkusen             341         2473
Monaco                       337         2447
AC Milan                     334         2847
Atletico Madrid              331         2418


In [11]:
h2h_counts = matches_df.groupby(['home_team', 'away_team']).size().reset_index(name='meetings')

# combine home+away perspective into one fixture pair (order-independent)
h2h_counts['fixture_pair'] = h2h_counts.apply(
    lambda r: tuple(sorted([r['home_team'], r['away_team']])), axis=1
)

h2h_summary = h2h_counts.groupby('fixture_pair')['meetings'].sum().sort_values(ascending=False)
print(h2h_summary.head(15))

fixture_pair
(AC Milan, Atalanta)                         10
(Inter, Lazio)                               10
(Chelsea, Tottenham)                         10
(Chelsea, West Ham)                          10
(Chelsea, Wolverhampton Wanderers)           10
(Crystal Palace, Everton)                    10
(Crystal Palace, Liverpool)                  10
(Crystal Palace, Manchester City)            10
(Crystal Palace, Manchester United)          10
(Crystal Palace, Newcastle United)           10
(Crystal Palace, Tottenham)                  10
(Crystal Palace, West Ham)                   10
(Crystal Palace, Wolverhampton Wanderers)    10
(Eintracht Frankfurt, Freiburg)              10
(Eintracht Frankfurt, Hoffenheim)            10
Name: meetings, dtype: int64


In [12]:
# build head-to-head goals: for each match, tag both teams' goals against each other
h2h_matches = matches_df[['home_team', 'away_team', 'home_goals', 'away_goals', 'league', 'year']].copy()

h2h_matches['fixture_pair'] = h2h_matches.apply(
    lambda r: tuple(sorted([r['home_team'], r['away_team']])), axis=1
)

# unpivot into one row per team per match (so we can sum goals per team per pairing)
home_rows = h2h_matches.rename(columns={'home_team': 'team_name', 'away_team': 'opponent', 'home_goals': 'goals_scored'})[['team_name', 'opponent', 'goals_scored', 'fixture_pair']]
away_rows = h2h_matches.rename(columns={'away_team': 'team_name', 'home_team': 'opponent', 'away_goals': 'goals_scored'})[['team_name', 'opponent', 'goals_scored', 'fixture_pair']]

h2h_long = pd.concat([home_rows, away_rows], ignore_index=True)

h2h_goals = h2h_long.groupby(['team_name', 'opponent', 'fixture_pair']).agg(
    goals_scored=('goals_scored', 'sum'),
    meetings=('goals_scored', 'count')
).reset_index()

h2h_goals['goals_per_meeting'] = (h2h_goals['goals_scored'] / h2h_goals['meetings']).round(2)

print(h2h_goals.sort_values('goals_scored', ascending=False).head(15))

                team_name                 opponent  \
2169  Paris Saint Germain              Montpellier   
365         Bayern Munich                   Bochum   
485     Borussia Dortmund                 Freiburg   
377         Bayern Munich                 Mainz 05   
1668            Liverpool                Tottenham   
382         Bayern Munich            VfB Stuttgart   
1805      Manchester City         Newcastle United   
2163  Paris Saint Germain                    Lille   
1855            Marseille              Montpellier   
1814      Manchester City  Wolverhampton Wanderers   
331             Barcelona               Real Betis   
337             Barcelona                 Valencia   
2176  Paris Saint Germain               Strasbourg   
480     Borussia Dortmund      Borussia M.Gladbach   
1662            Liverpool        Manchester United   

                                    fixture_pair  goals_scored  meetings  \
2169          (Montpellier, Paris Saint Germain)           

In [13]:
players_df = pd.read_csv('../data/processed/players.csv')
print(players_df.columns.tolist())


['id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'position', 'team_title', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'league', 'year']


In [14]:
players_df['primary_position'] = players_df['position'].str.split().str[0]
print(players_df['primary_position'].value_counts())

primary_position
D     5166
F     3155
M     2865
S     1795
GK     983
Name: count, dtype: int64


In [15]:
print(players_df.columns.tolist())

['id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'position', 'team_title', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'league', 'year', 'primary_position']


In [16]:
players_df['per90_reliable'] = players_df['time'] >= 450
players_df['npxG_per90'] = (players_df['npxG'] / players_df['time'] * 90).round(3)
players_df['goals_per90'] = (players_df['goals'] / players_df['time'] * 90).round(3)
players_df['xGChain_per90'] = (players_df['xGChain'] / players_df['time'] * 90).round(3)
players_df['xA_per90'] = (players_df['xA'] / players_df['time'] * 90).round(3)
players_df['key_passes_per90'] = (players_df['key_passes'] / players_df['time'] * 90).round(3)

print(players_df.columns.tolist())

['id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'position', 'team_title', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'league', 'year', 'primary_position', 'per90_reliable', 'npxG_per90', 'goals_per90', 'xGChain_per90', 'xA_per90', 'key_passes_per90']


In [17]:
anomaly_pool = players_df[(players_df['per90_reliable']) & (players_df['primary_position'] != 'S')].copy()

def zscore_within_group(df, metric, group_cols):
    stats = df.groupby(group_cols)[metric].agg(['mean', 'std'])
    merged = df.merge(stats, on=group_cols, suffixes=('', '_stat'))
    merged[f'{metric}_zscore'] = (merged[metric] - merged['mean']) / merged['std']
    return merged.drop(columns=['mean', 'std'])

metrics = ['npxG_per90', 'goals_per90', 'xGChain_per90', 'xA_per90', 'key_passes_per90']
for m in metrics:
    anomaly_pool = zscore_within_group(anomaly_pool, m, ['primary_position', 'league', 'year'])

print(anomaly_pool[anomaly_pool['npxG_per90_zscore'] > 2].sort_values('npxG_per90_zscore', ascending=False)[['player_name', 'team_title', 'league', 'year', 'primary_position', 'npxG_per90', 'npxG_per90_zscore']].head(15))

              player_name         team_title          league  year  \
4699    Zakaria Aboukhlal           Toulouse         ligue_1  2022   
7143   Keane Lewis-Potter          Brentford  premier_league  2023   
9118      Davide Frattesi              Inter         serie_a  2023   
6251    Christian Pulisic            Chelsea  premier_league  2021   
7460         Noni Madueke            Chelsea  premier_league  2024   
6159  Alireza Jahanbakhsh           Brighton  premier_league  2020   
1496        Donyell Malen  Borussia Dortmund      bundesliga  2024   
1865    Philippe Coutinho          Barcelona         la_liga  2020   
6719   Alejandro Garnacho  Manchester United  premier_league  2022   
3541          Samuel Lino    Atletico Madrid         la_liga  2024   
5132         Yusuf Yazici              Lille         ligue_1  2023   
361          Serge Gnabry      Bayern Munich      bundesliga  2021   
3037     Cristhian Stuani             Girona         la_liga  2023   
3806          Joan G

In [18]:
check_players = ['Zakaria Aboukhlal', 'Christian Pulisic', 'Noni Madueke', 'Serge Gnabry']
raw_check = players_df[players_df['player_name'].isin(check_players)]
print(raw_check[['player_name', 'position', 'primary_position']].to_string())

             player_name position primary_position
11          Serge Gnabry      M S                M
502         Serge Gnabry    D M S                D
1011        Serge Gnabry    F M S                F
1609        Serge Gnabry      M S                M
2042        Serge Gnabry    F M S                F
6618   Zakaria Aboukhlal  D F M S                D
7278   Zakaria Aboukhlal    F M S                F
7736   Zakaria Aboukhlal    F M S                F
8322   Christian Pulisic    F M S                F
8820   Christian Pulisic  D F M S                D
9514   Christian Pulisic  D F M S                D
9577        Noni Madueke    F M S                F
9947        Noni Madueke      M S                M
10484       Noni Madueke    D M S                D
12785  Christian Pulisic    F M S                F
13375  Christian Pulisic    F M S                F


In [19]:
print(players_df[players_df['player_name'] == 'Serge Gnabry'][['player_name', 'league', 'year', 'team_title', 'time', 'position']])

       player_name      league  year     team_title  time position
11    Serge Gnabry  bundesliga  2020  Bayern Munich  1653      M S
502   Serge Gnabry  bundesliga  2021  Bayern Munich  2205    D M S
1011  Serge Gnabry  bundesliga  2022  Bayern Munich  1950    F M S
1609  Serge Gnabry  bundesliga  2023  Bayern Munich   431      M S
2042  Serge Gnabry  bundesliga  2024  Bayern Munich  1216    F M S


In [20]:
def resolve_position(pos_string):
    tokens = set(pos_string.split())
    if 'GK' in tokens:
        return 'GK'
    elif 'F' in tokens:
        return 'F'
    elif 'M' in tokens:
        return 'M'
    elif 'D' in tokens:
        return 'D'
    else:
        return 'S'

players_df['primary_position_hierarchy'] = players_df['position'].apply(resolve_position)
print(players_df['primary_position_hierarchy'].value_counts())

print(players_df[players_df['player_name'] == 'Serge Gnabry'][['player_name', 'year', 'position', 'primary_position_hierarchy']])

primary_position_hierarchy
M     4174
D     3548
F     3464
S     1795
GK     983
Name: count, dtype: int64
       player_name  year position primary_position_hierarchy
11    Serge Gnabry  2020      M S                          M
502   Serge Gnabry  2021    D M S                          M
1011  Serge Gnabry  2022    F M S                          F
1609  Serge Gnabry  2023      M S                          M
2042  Serge Gnabry  2024    F M S                          F


In [22]:
# aggregate actual minutes played per position, per player-season, from rosters_df
rosters_df = pd.read_csv("/Users/aadvikmazumdar/Projects/OnTarget/data/processed/rosters.csv")
minutes_by_position = rosters_df.groupby(['player_id', 'league', 'year', 'position'])['time'].sum().reset_index()

# map rosters_df's detailed codes (DC, DL, DML etc.) to the same broad buckets
def broad_position(code):
    if code == 'GK':
        return 'GK'
    elif code.startswith('D'):
        return 'D'
    elif code.startswith('AM') or code in ['MC', 'ML', 'MR']:
        return 'M'
    elif code.startswith('FW'):
        return 'F'
    else:
        return 'Other'

minutes_by_position['broad_position'] = minutes_by_position['position'].apply(broad_position)

# sum minutes by broad position, then pick whichever bucket has the MOST actual minutes
minutes_summed = minutes_by_position.groupby(['player_id', 'league', 'year', 'broad_position'])['time'].sum().reset_index()

primary_by_minutes = minutes_summed.loc[
    minutes_summed.groupby(['player_id', 'league', 'year'])['time'].idxmax()
][['player_id', 'league', 'year', 'broad_position']].rename(columns={'broad_position': 'primary_position_minutes'})

print(primary_by_minutes.head(10))

    player_id      league  year primary_position_minutes
0           3     serie_a  2020                        D
2           3     serie_a  2022                        D
4           3     serie_a  2023                        D
6           3     serie_a  2024                        D
9           5  bundesliga  2023                        M
12          9  bundesliga  2023                        M
14         19  bundesliga  2020                       GK
15         22  bundesliga  2020                        D
17         22  bundesliga  2021                        D
19         22  bundesliga  2022                        D


In [23]:
# merge players_df id column with rosters_df's player_id — confirm they're the same key first
print(players_df[['id', 'player_name']].head())
print(rosters_df[['player_id', 'player']].head())

     id         player_name
0   227  Robert Lewandowski
1  6170         André Silva
2  8260      Erling Haaland
3   956     Andrej Kramaric
4  7052       Wout Weghorst
   player_id          player
0       8715   Illan Meslier
1       8716     Luke Ayling
2       8816     Liam Cooper
3       6273      Robin Koch
4       8722  Ezgjan Alioski


In [24]:
players_ids = set(players_df['id'].unique())
rosters_ids = set(rosters_df['player_id'].unique())

overlap = players_ids & rosters_ids
print(f"players_df unique ids: {len(players_ids)}")
print(f"rosters_df unique ids: {len(rosters_ids)}")
print(f"overlapping ids: {len(overlap)}")

players_df unique ids: 5630
rosters_df unique ids: 5630
overlapping ids: 5630


In [25]:
comparison = players_df[['id', 'player_name', 'league', 'year', 'primary_position_hierarchy']].merge(
    primary_by_minutes.rename(columns={'player_id': 'id'}),
    on=['id', 'league', 'year'],
    how='inner'
)

comparison['agree'] = comparison['primary_position_hierarchy'] == comparison['primary_position_minutes']

print(comparison['agree'].value_counts())
print()
print(comparison['agree'].value_counts(normalize=True).round(3))

agree
True     8720
False    5244
Name: count, dtype: int64

agree
True     0.624
False    0.376
Name: proportion, dtype: float64


In [26]:
players_df = players_df.merge(
    primary_by_minutes.rename(columns={'player_id': 'id'}),
    on=['id', 'league', 'year'],
    how='left'
)

print(players_df['primary_position_minutes'].isna().sum(), 'unmatched players')

anomaly_pool = players_df[(players_df['per90_reliable']) & (players_df['primary_position_minutes'].notna()) & (players_df['primary_position_minutes'] != 'Other')].copy()

def zscore_within_group(df, metric, group_cols):
    stats = df.groupby(group_cols)[metric].agg(['mean', 'std'])
    merged = df.merge(stats, on=group_cols, suffixes=('', '_stat'))
    merged[f'{metric}_zscore'] = (merged[metric] - merged['mean']) / merged['std']
    return merged.drop(columns=['mean', 'std'])

metrics = ['npxG_per90', 'goals_per90', 'xGChain_per90', 'xA_per90', 'key_passes_per90']
for m in metrics:
    anomaly_pool = zscore_within_group(anomaly_pool, m, ['primary_position_minutes', 'league', 'year'])

print(anomaly_pool[anomaly_pool['npxG_per90_zscore'] > 2].sort_values('npxG_per90_zscore', ascending=False)[['player_name', 'team_title', 'league', 'year', 'primary_position_minutes', 'npxG_per90', 'npxG_per90_zscore']].head(15))

0 unmatched players
            player_name           team_title          league  year  \
7389       Ross Barkley          Aston Villa  premier_league  2024   
8961    Davide Frattesi                Inter         serie_a  2023   
1828  Philippe Coutinho            Barcelona         la_liga  2020   
2652  Ander Barrenetxea        Real Sociedad         la_liga  2022   
6251       Matt Doherty            Tottenham  premier_league  2021   
1056   Jeremie Frimpong     Bayer Leverkusen      bundesliga  2023   
6563        Deniz Undav             Brighton  premier_league  2022   
1092       Nathan Tella     Bayer Leverkusen      bundesliga  2023   
4306       Sergio Ramos  Paris Saint Germain         ligue_1  2021   
404       Jamal Musiala        Bayern Munich      bundesliga  2021   
3735        Joan García             Espanyol         la_liga  2024   
7575               Kepa          Bournemouth  premier_league  2024   
2924   Jeremías Ledesma                Cadiz         la_liga  2022   


In [27]:
anomaly_pool = players_df[
    (players_df['per90_reliable']) &
    (players_df['primary_position_minutes'].notna()) &
    (~players_df['primary_position_minutes'].isin(['Other', 'GK']))
].copy()

for m in metrics:
    anomaly_pool = zscore_within_group(anomaly_pool, m, ['primary_position_minutes', 'league', 'year'])

print(anomaly_pool[anomaly_pool['npxG_per90_zscore'] > 2].sort_values('npxG_per90_zscore', ascending=False)[['player_name', 'team_title', 'league', 'year', 'primary_position_minutes', 'npxG_per90', 'npxG_per90_zscore']].head(15))

            player_name           team_title          league  year  \
6860       Ross Barkley          Aston Villa  premier_league  2024   
8309    Davide Frattesi                Inter         serie_a  2023   
1702  Philippe Coutinho            Barcelona         la_liga  2020   
2467  Ander Barrenetxea        Real Sociedad         la_liga  2022   
5808       Matt Doherty            Tottenham  premier_league  2021   
983    Jeremie Frimpong     Bayer Leverkusen      bundesliga  2023   
6094        Deniz Undav             Brighton  premier_league  2022   
1019       Nathan Tella     Bayer Leverkusen      bundesliga  2023   
4000       Sergio Ramos  Paris Saint Germain         ligue_1  2021   
383       Jamal Musiala        Bayern Munich      bundesliga  2021   
3961       Pereira Lage               Angers         ligue_1  2021   
8507          Emil Holm             Atalanta         serie_a  2023   
3533             Neymar  Paris Saint Germain         ligue_1  2020   
7144       Robin Gos

In [28]:
print(anomaly_pool.groupby('primary_position_minutes')['npxG_per90'].agg(['mean', 'std', 'count']).round(4))

                            mean     std  count
primary_position_minutes                       
D                         0.0529  0.0463   4270
F                         0.3560  0.1653   1540
M                         0.1402  0.1082   3211


In [29]:
players_df.to_csv('../data/processed/players_enriched.csv', index=False)
print("saved players_enriched.csv with shape:", players_df.shape)

saved players_enriched.csv with shape: (13964, 29)


In [30]:
player_season_counts = players_df.groupby('id')['year'].nunique()
multi_season_players = player_season_counts[player_season_counts >= 3].index

peak_pool = players_df[
    (players_df['id'].isin(multi_season_players)) &
    (players_df['per90_reliable'])
].copy()

print(f"Players with 3+ reliable seasons: {peak_pool['id'].nunique()}")
print(peak_pool.shape)

Players with 3+ reliable seasons: 2215
(7739, 29)


In [31]:
def player_own_zscore(df, metric):
    stats = df.groupby('id')[metric].agg(['mean', 'std']).rename(columns={'mean': f'{metric}_own_mean', 'std': f'{metric}_own_std'})
    merged = df.merge(stats, on='id')
    merged[f'{metric}_own_zscore'] = (merged[metric] - merged[f'{metric}_own_mean']) / merged[f'{metric}_own_std']
    return merged

peak_pool = player_own_zscore(peak_pool, 'npxG_per90')

peak_pool['peak_season'] = peak_pool['npxG_per90_own_zscore'] > 1.5

print(peak_pool['peak_season'].sum(), 'peak seasons flagged')
print()
print(peak_pool[peak_pool['peak_season']].sort_values('npxG_per90_own_zscore', ascending=False)[
    ['player_name', 'team_title', 'league', 'year', 'npxG_per90', 'npxG_per90_own_mean', 'npxG_per90_own_zscore']
].head(15))

207 peak seasons flagged

                 player_name               team_title          league  year  \
3522               Matz Sels               Strasbourg         ligue_1  2021   
2465                Reinildo          Atletico Madrid         la_liga  2023   
4230           Marco Asensio      Paris Saint Germain         ligue_1  2024   
4757         Bruno Guimarães         Newcastle United  premier_league  2021   
1564   Marc-André ter Stegen                Barcelona         la_liga  2020   
5674               Nick Pope         Newcastle United  premier_league  2023   
1529         Marko Dmitrovic                    Eibar         la_liga  2020   
6657  Vanja Milinkovic-Savic                   Torino         serie_a  2021   
1885               Jan Oblak          Atletico Madrid         la_liga  2021   
4534                 Alisson                Liverpool  premier_league  2020   
785             Manuel Neuer            Bayern Munich      bundesliga  2022   
6549           Martin Hong

In [32]:
peak_pool = players_df[
    (players_df['id'].isin(multi_season_players)) &
    (players_df['per90_reliable']) &
    (players_df['primary_position_minutes'] != 'GK')
].copy()

peak_pool = player_own_zscore(peak_pool, 'npxG_per90')
peak_pool['peak_season'] = peak_pool['npxG_per90_own_zscore'] > 1.5

print(peak_pool['peak_season'].sum(), 'peak seasons flagged')
print()
print(peak_pool[peak_pool['peak_season']].sort_values('npxG_per90_own_zscore', ascending=False)[
    ['player_name', 'team_title', 'league', 'year', 'npxG_per90', 'npxG_per90_own_mean', 'npxG_per90_own_zscore']
].head(15))

197 peak seasons flagged

           player_name               team_title          league  year  \
2295          Reinildo          Atletico Madrid         la_liga  2023   
3941     Marco Asensio      Paris Saint Germain         ligue_1  2024   
4429   Bruno Guimarães         Newcastle United  premier_league  2021   
6101     Martin Hongla                   Verona         serie_a  2021   
1005        Kevin Vogt  Hoffenheim,Union Berlin      bundesliga  2023   
3672   Youssouf Fofana                   Monaco         ligue_1  2023   
2041         Loic Bade                  Sevilla         la_liga  2022   
6045   Davide Calabria                 AC Milan         serie_a  2021   
4706      Granit Xhaka                  Arsenal  premier_league  2022   
5382      Mikel Merino                  Arsenal  premier_league  2024   
65            Angelino   RasenBallsport Leipzig      bundesliga  2020   
6250       Rafael Leão                 AC Milan         serie_a  2022   
3152  Ibrahima Sissoko   

In [33]:
print(peak_pool.groupby('primary_position_minutes')['npxG_per90'].agg(['mean', 'std', 'count']).round(4))

                            mean     std  count
primary_position_minutes                       
D                         0.0556  0.0479   3359
F                         0.3659  0.1677   1210
M                         0.1479  0.1116   2505
Other                     0.2545  0.1884    122


In [34]:
players_df['penalty_dependency'] = (players_df['xG'] - players_df['npxG']).round(3)

penalty_reliant = players_df[players_df['per90_reliable']].sort_values('penalty_dependency', ascending=False)
print(penalty_reliant[['player_name', 'team_title', 'league', 'year', 'goals', 'xG', 'npxG', 'penalty_dependency']].head(15))

                    player_name               team_title          league  \
11011             Franck Kessié                 AC Milan         serie_a   
11607           Lorenzo Insigne                   Napoli         serie_a   
5435          Wissam Ben Yedder                   Monaco         ligue_1   
6599   Jonathan Christian David                    Lille         ligue_1   
3058              Karim Benzema              Real Madrid         la_liga   
12780          Hakan Calhanoglu                    Inter         serie_a   
8251            Bruno Fernandes        Manchester United  premier_league   
2489              Gerard Moreno               Villarreal         la_liga   
11590             Ciro Immobile                    Lazio         serie_a   
10434             Mohamed Salah                Liverpool  premier_league   
8255                Jamie Vardy                Leicester  premier_league   
9865                Cole Palmer  Chelsea,Manchester City  premier_league   
8293        

In [35]:
players_df['penalty_dependency_pct'] = (players_df['penalty_dependency'] / players_df['xG'].replace(0, np.nan) * 100).round(1)

print(players_df[players_df['per90_reliable']].sort_values('penalty_dependency_pct', ascending=False)[
    ['player_name', 'team_title', 'league', 'year', 'xG', 'npxG', 'penalty_dependency_pct']
].head(15))

             player_name         team_title          league  year        xG  \
1699    Leonardo Bonucci       Union Berlin      bundesliga  2023  0.757777   
2177          Kevin Vogt       Union Berlin      bundesliga  2024  0.757777   
316       Nabil Bentaleb         Schalke 04      bundesliga  2020  0.757777   
944       Manuel Riemann             Bochum      bundesliga  2021  0.757777   
8293            Jorginho            Chelsea  premier_league  2020  6.972691   
2716     Marko Dmitrovic              Eibar         la_liga  2020  1.558895   
11644  Domenico Criscito              Genoa         serie_a  2021  5.659400   
540             Emre Can  Borussia Dortmund      bundesliga  2021  3.246970   
8817            Jorginho            Chelsea  premier_league  2021  5.913266   
5537          Kenny Lala         Strasbourg         ligue_1  2020  2.586738   
11226  Domenico Criscito              Genoa         serie_a  2020  0.868794   
4278              Pepelu           Valencia         

In [36]:
penalty_rate_filtered = players_df[(players_df['per90_reliable']) & (players_df['xG'] >= 3.0)].sort_values('penalty_dependency_pct', ascending=False)

print(penalty_rate_filtered[['player_name', 'team_title', 'league', 'year', 'xG', 'npxG', 'penalty_dependency_pct']].head(15))

             player_name         team_title          league  year         xG  \
8293            Jorginho            Chelsea  premier_league  2020   6.972691   
11644  Domenico Criscito              Genoa         serie_a  2021   5.659400   
540             Emre Can  Borussia Dortmund      bundesliga  2021   3.246970   
8817            Jorginho            Chelsea  premier_league  2021   5.913266   
4278              Pepelu           Valencia         la_liga  2023   6.946358   
11069      Nicolas Viola          Benevento         serie_a  2020   3.597570   
6049      Thomas Mangani             Angers         ligue_1  2021   5.437245   
6100     Florian Tardieu             Troyes         ligue_1  2021   3.686876   
1062   Maximilian Arnold          Wolfsburg      bundesliga  2022   4.634974   
1564       Florian Kainz         FC Cologne      bundesliga  2023   4.688287   
12780   Hakan Calhanoglu              Inter         serie_a  2023   9.977345   
12869    Leandro Paredes               R

In [37]:
print(shots_df[['player_id', 'match_id', 'h_a']].head())
print(rosters_df[['player_id', 'match_id', 'h_a', 'position']].head())

   player_id  match_id h_a
0       6018     13977   a
1       7169     13977   a
2       6018     13977   a
3       6902     13977   h
4       6060     13977   h
   player_id  match_id h_a position
0       8715     14518   h       GK
1       8716     14518   h       DR
2       8816     14518   h       DC
3       6273     14518   h       DC
4       8722     14518   h       DL


In [38]:
shots_with_position = shots_df.merge(
    rosters_df[['player_id', 'match_id', 'h_a', 'position']],
    on=['player_id', 'match_id', 'h_a'],
    how='inner'
)

print(shots_with_position.shape, shots_df.shape)

# broad position bucket, same logic as before
def broad_position(code):
    if code == 'GK':
        return 'GK'
    elif code.startswith('D'):
        return 'D'
    elif code.startswith('AM') or code in ['MC', 'ML', 'MR']:
        return 'M'
    elif code.startswith('FW'):
        return 'F'
    else:
        return 'Other'

shots_with_position['broad_position'] = shots_with_position['position'].apply(broad_position)

positional_evolution = shots_with_position.groupby(['broad_position', 'year']).agg(
    shots=('X', 'count'),
    avg_shot_distance=('X', 'mean')
).round(3)

print(positional_evolution)

(224699, 39) (224676, 38)
                     shots  avg_shot_distance
broad_position year                          
D              2020   9364              0.842
               2021   9904              0.845
               2022  10290              0.844
               2023  10494              0.844
               2024  10310              0.845
F              2020  12934              0.871
               2021  13822              0.870
               2022  13429              0.872
               2023  12553              0.875
               2024  10880              0.876
GK             2020     30              0.348
               2021     24              0.317
               2022     22              0.356
               2023     28              0.335
               2024     22              0.378
M              2020  15268              0.833
               2021  15632              0.832
               2022  15052              0.835
               2023  15811              0.838
        

In [39]:
zone_evolution = shots_with_position.groupby(['broad_position', 'year'])['shot_zone'].value_counts(normalize=True).unstack().round(3)
print(zone_evolution)

shot_zone            edge_of_box  outside_box  six_yard_box
broad_position year                                        
D              2020        0.360        0.044         0.596
               2021        0.350        0.045         0.606
               2022        0.356        0.042         0.602
               2023        0.372        0.039         0.589
               2024        0.366        0.039         0.594
F              2020        0.276        0.013         0.711
               2021        0.279        0.014         0.707
               2022        0.274        0.011         0.715
               2023        0.259        0.010         0.731
               2024        0.251        0.009         0.740
GK             2020        0.133        0.633         0.233
               2021        0.083        0.667         0.250
               2022          NaN        0.636         0.364
               2023          NaN        0.643         0.357
               2024          NaN        

In [40]:
side_evolution = shots_with_position.groupby(['broad_position', 'year'])['cross_side'].value_counts(normalize=True).unstack().round(3)
print(side_evolution)

cross_side            left  right
broad_position year              
D              2020  0.482  0.518
               2021  0.486  0.514
               2022  0.481  0.519
               2023  0.490  0.510
               2024  0.487  0.513
F              2020  0.465  0.535
               2021  0.471  0.529
               2022  0.474  0.526
               2023  0.473  0.527
               2024  0.474  0.526
GK             2020  0.533  0.467
               2021  0.583  0.417
               2022  0.636  0.364
               2023  0.429  0.571
               2024  0.545  0.455
M              2020  0.472  0.528
               2021  0.485  0.515
               2022  0.490  0.510
               2023  0.488  0.512
               2024  0.488  0.512
Other          2020  0.480  0.520
               2021  0.486  0.514
               2022  0.477  0.523
               2023  0.485  0.515
               2024  0.490  0.510


In [41]:
def y_zone_five(y):
    if y < 0.2:
        return 'extreme_left'
    elif y < 0.4:
        return 'mid_left'
    elif y < 0.6:
        return 'centre'
    elif y < 0.8:
        return 'mid_right'
    else:
        return 'extreme_right'

shots_with_position['y_zone_5'] = shots_with_position['Y'].apply(y_zone_five)

print(shots_with_position['y_zone_5'].value_counts(normalize=True).round(3))

y_zone_5
centre           0.552
mid_right        0.231
mid_left         0.209
extreme_right    0.004
extreme_left     0.003
Name: proportion, dtype: float64


In [43]:
player_shot_profile_season = shots_with_position.groupby(['player', 'year', 'y_zone_5']).size().unstack(fill_value=0)
player_shot_totals_season = player_shot_profile_season.sum(axis=1)

reliable_player_seasons = player_shot_totals_season[player_shot_totals_season >= 30].index
player_shot_pct_season = player_shot_profile_season.loc[reliable_player_seasons].div(player_shot_totals_season.loc[reliable_player_seasons], axis=0).round(3)

print(player_shot_pct_season.head(10))

y_zone_5                    centre  extreme_left  extreme_right  mid_left  \
player                year                                                  
Abdallah Sima         2022   0.600           0.0          0.000     0.150   
                      2024   0.711           0.0          0.000     0.026   
Abdelhamid Sabiri     2022   0.485           0.0          0.000     0.212   
Abdessamad Ezzalzouli 2022   0.150           0.0          0.000     0.050   
                      2023   0.263           0.0          0.000     0.026   
                      2024   0.468           0.0          0.016     0.016   
Abdou Harroui         2023   0.485           0.0          0.000     0.061   
Abdoulaye Doucouré    2021   0.529           0.0          0.000     0.294   
                      2023   0.702           0.0          0.000     0.085   
                      2024   0.484           0.0          0.000     0.194   

y_zone_5                    mid_right  
player                year         

In [44]:
if 'team_name' not in shots_with_position.columns:
    shots_with_position['team_name'] = np.where(
        shots_with_position['h_a'] == 'h',
        shots_with_position['h_team'],
        shots_with_position['a_team']
    )

team_shot_profile_season = shots_with_position.groupby(['team_name', 'year', 'y_zone_5']).size().unstack(fill_value=0)
team_shot_totals_season = team_shot_profile_season.sum(axis=1)

reliable_team_seasons = team_shot_totals_season[team_shot_totals_season >= 50].index
team_shot_pct_season = team_shot_profile_season.loc[reliable_team_seasons].div(team_shot_totals_season.loc[reliable_team_seasons], axis=0).round(3)

print(team_shot_pct_season.head(10))

y_zone_5        centre  extreme_left  extreme_right  mid_left  mid_right
team_name year                                                          
AC Milan  2020   0.579         0.000          0.005     0.165      0.251
          2021   0.552         0.002          0.005     0.202      0.240
          2022   0.564         0.002          0.002     0.186      0.246
          2023   0.567         0.000          0.005     0.179      0.248
          2024   0.575         0.000          0.007     0.196      0.222
Ajaccio   2022   0.607         0.000          0.009     0.221      0.162
Alaves    2020   0.606         0.003          0.003     0.167      0.221
          2021   0.534         0.000          0.000     0.241      0.225
          2023   0.548         0.009          0.013     0.199      0.231
          2024   0.583         0.002          0.007     0.193      0.215


In [45]:
zone_conversion = shots_with_position.groupby('y_zone_5').agg(
    shots=('y_zone_5', 'count'),
    goals=('result', lambda x: (x == 'Goal').sum()),
    avg_xG=('xG', 'mean')
).assign(
    conversion=lambda d: d['goals']/d['shots']
).round(4)

print(zone_conversion)

                shots  goals  avg_xG  conversion
y_zone_5                                        
centre         124139  18782  0.1621      0.1513
extreme_left      728     52  0.0321      0.0714
extreme_right     937     59  0.0321      0.0630
mid_left        47055   2732  0.0643      0.0581
mid_right       51840   2903  0.0639      0.0560


In [46]:
zone_conversion_by_position = shots_with_position.groupby(['broad_position', 'y_zone_5']).agg(
    shots=('y_zone_5', 'count'),
    goals=('result', lambda x: (x == 'Goal').sum()),
    avg_xG=('xG', 'mean')
).assign(conversion=lambda d: d['goals']/d['shots']).round(4)

print(zone_conversion_by_position)

                              shots  goals  avg_xG  conversion
broad_position y_zone_5                                       
D              centre         29154   3014  0.1177      0.1034
               extreme_left     169      8  0.0318      0.0473
               extreme_right    186     10  0.0255      0.0538
               mid_left        9837    450  0.0475      0.0457
               mid_right      11016    429  0.0462      0.0389
F              centre         37119   7430  0.2131      0.2002
               extreme_left     145     12  0.0279      0.0828
               extreme_right    212     11  0.0323      0.0519
               mid_left       12227    865  0.0830      0.0707
               mid_right      13915    997  0.0829      0.0716
GK             centre           112      3  0.0418      0.0268
               mid_left           3      0  0.0110      0.0000
               mid_right         11      0  0.0250      0.0000
M              centre         39800   5594  0.1450     

In [47]:
print('age' in players_df.columns or 'birth_date' in players_df.columns)
print(players_df.columns.tolist())

False
['id', 'player_name', 'games', 'time', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'position', 'team_title', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'league', 'year', 'primary_position', 'per90_reliable', 'npxG_per90', 'goals_per90', 'xGChain_per90', 'xA_per90', 'key_passes_per90', 'primary_position_hierarchy', 'primary_position_minutes', 'penalty_dependency', 'penalty_dependency_pct']


In [48]:
trajectory_pool = players_df[
    (players_df['per90_reliable']) & 
    (players_df['primary_position_minutes'] != 'GK')
].sort_values(['id', 'year']).copy()

def classify_trajectory(group):
    group = group.sort_values('year')
    if len(group) < 3:
        return 'insufficient_data'
    
    diffs = group['npxG_per90'].diff().dropna().values
    
    if len(diffs) >= 3:
        last_three = diffs[-3:]
    else:
        last_three = diffs
    
    if (last_three > 0).all():
        return 'ascending'
    elif (last_three < 0).all():
        return 'declining'
    else:
        return 'established'

trajectories = trajectory_pool.groupby('id').apply(classify_trajectory, include_groups=False)
print(trajectories.value_counts())

insufficient_data    2000
established          1277
ascending             173
declining             152
Name: count, dtype: int64


In [49]:
ascending_ids = trajectories[trajectories == 'ascending'].index
sample_ascending = players_df[players_df['id'].isin(ascending_ids)].sort_values(['id', 'year'])
print(sample_ascending[sample_ascending['id'].isin(ascending_ids[:5])][['player_name', 'year', 'npxG_per90']])

                    player_name  year  npxG_per90
154             Mitchell Weiser  2020       0.033
1135            Mitchell Weiser  2022       0.118
1601            Mitchell Weiser  2023       0.130
2057            Mitchell Weiser  2024       0.174
6                   Lars Stindl  2020       0.177
556                 Lars Stindl  2021       0.250
1032                Lars Stindl  2022       0.297
91                  Nico Elvedi  2020       0.084
683                 Nico Elvedi  2021       0.049
1110                Nico Elvedi  2022       0.070
1634                Nico Elvedi  2023       0.072
2181                Nico Elvedi  2024       0.077
93     Eric Maxim Choupo-Moting  2020       0.473
558    Eric Maxim Choupo-Moting  2021       1.212
1022   Eric Maxim Choupo-Moting  2022       0.606
1635   Eric Maxim Choupo-Moting  2023       0.730
11126               Albin Ekdal  2020       0.034
11762               Albin Ekdal  2021       0.041
12484               Albin Ekdal  2022       0.141


In [50]:
print(players_df[players_df['player_name'] == 'Eric Maxim Choupo-Moting'][['player_name', 'year', 'npxG_per90']])
print(trajectories.get(players_df[players_df['player_name'] == 'Eric Maxim Choupo-Moting']['id'].iloc[0]))

                   player_name  year  npxG_per90
93    Eric Maxim Choupo-Moting  2020       0.473
558   Eric Maxim Choupo-Moting  2021       1.212
1022  Eric Maxim Choupo-Moting  2022       0.606
1635  Eric Maxim Choupo-Moting  2023       0.730
ascending


In [51]:
test_id = players_df[players_df['player_name'] == 'Eric Maxim Choupo-Moting']['id'].iloc[0]
test_group = trajectory_pool[trajectory_pool['id'] == test_id].sort_values('year')
print(test_group[['year', 'npxG_per90']])

diffs = test_group['npxG_per90'].diff().dropna().values
print('diffs:', diffs)
print('last_three:', diffs[-3:])
print('all positive?', (diffs[-3:] > 0).all())

      year  npxG_per90
93    2020       0.473
1022  2022       0.606
1635  2023       0.730
diffs: [0.133 0.124]
last_three: [0.133 0.124]
all positive? True


In [52]:
def classify_trajectory(group):
    group = group.sort_values('year')
    years = group['year'].values
    
    if len(group) < 3:
        return 'insufficient_data'
    
    # check the years are consecutive (gap of exactly 1 between each)
    year_gaps = np.diff(years)
    if not (year_gaps == 1).all():
        return 'non_consecutive_seasons'
    
    diffs = group['npxG_per90'].diff().dropna().values
    last_three = diffs[-3:] if len(diffs) >= 3 else diffs
    
    if (last_three > 0).all():
        return 'ascending'
    elif (last_three < 0).all():
        return 'declining'
    else:
        return 'established'

trajectories = trajectory_pool.groupby('id').apply(classify_trajectory, include_groups=False)
print(trajectories.value_counts())

insufficient_data          2000
established                1023
non_consecutive_seasons     327
ascending                   129
declining                   123
Name: count, dtype: int64


In [53]:
print(trajectories.get(test_id))

non_consecutive_seasons


In [54]:
ascending_ids = trajectories[trajectories == 'ascending'].index
sample = players_df[players_df['id'].isin(list(ascending_ids)[:5])].sort_values(['id', 'year'])
print(sample[['player_name', 'year', 'npxG_per90']])

           player_name  year  npxG_per90
154    Mitchell Weiser  2020       0.033
1135   Mitchell Weiser  2022       0.118
1601   Mitchell Weiser  2023       0.130
2057   Mitchell Weiser  2024       0.174
6          Lars Stindl  2020       0.177
556        Lars Stindl  2021       0.250
1032       Lars Stindl  2022       0.297
91         Nico Elvedi  2020       0.084
683        Nico Elvedi  2021       0.049
1110       Nico Elvedi  2022       0.070
1634       Nico Elvedi  2023       0.072
2181       Nico Elvedi  2024       0.077
11126      Albin Ekdal  2020       0.034
11762      Albin Ekdal  2021       0.041
12484      Albin Ekdal  2022       0.141
1114       Mario Götze  2022       0.108
1608       Mario Götze  2023       0.121
2085       Mario Götze  2024       0.171


In [55]:
weiser_id = players_df[players_df['player_name'] == 'Mitchell Weiser']['id'].iloc[0]
print(trajectories.get(weiser_id))

ascending


In [56]:
weiser_group = trajectory_pool[trajectory_pool['id'] == weiser_id].sort_values('year')
print(weiser_group[['year', 'npxG_per90']])
print()
print('years array:', weiser_group['year'].values)
print('gaps:', np.diff(weiser_group['year'].values))

      year  npxG_per90
1135  2022       0.118
1601  2023       0.130
2057  2024       0.174

years array: [2022 2023 2024]
gaps: [1 1]


In [57]:
declining_ids = trajectories[trajectories == 'declining'].index
sample_declining = players_df[players_df['id'].isin(list(declining_ids)[:5])].sort_values(['id', 'year'])
print(sample_declining[['player_name', 'year', 'npxG_per90']])

              player_name  year  npxG_per90
69           Filip Kostic  2020       0.146
553          Filip Kostic  2021       0.106
1274         Filip Kostic  2022       0.021
12283        Filip Kostic  2022       0.092
13059        Filip Kostic  2023       0.034
5436        Kevin Volland  2020       0.355
6034        Kevin Volland  2021       0.280
6692        Kevin Volland  2022       0.272
1603        Kevin Volland  2023       0.228
2265        Kevin Volland  2024       0.000
89           Jonas Hector  2020       0.189
769          Jonas Hector  2021       0.047
1285         Jonas Hector  2022       0.040
8257       Ilkay Gündogan  2020       0.356
8800       Ilkay Gündogan  2021       0.472
9338       Ilkay Gündogan  2022       0.269
4297       Ilkay Gündogan  2023       0.172
10614      Ilkay Gündogan  2024       0.138
11008  Henrikh Mkhitaryan  2020       0.381
11651  Henrikh Mkhitaryan  2021       0.136
12285  Henrikh Mkhitaryan  2022       0.127
12896  Henrikh Mkhitaryan  2023 

In [58]:
player_year_counts = players_df.groupby(['id', 'year']).size()
multi_row_seasons = player_year_counts[player_year_counts > 1]
print(f"{len(multi_row_seasons)} player-year combinations with multiple rows")
print(f"out of {len(player_year_counts)} total player-year combinations")

print()
print(players_df[players_df['id'] == players_df[players_df['player_name'] == 'Filip Kostic']['id'].iloc[0]][['player_name', 'year', 'team_title', 'time', 'npxG_per90']])

322 player-year combinations with multiple rows
out of 13641 total player-year combinations

        player_name  year           team_title  time  npxG_per90
69     Filip Kostic  2020  Eintracht Frankfurt  2535       0.146
553    Filip Kostic  2021  Eintracht Frankfurt  2527       0.106
1274   Filip Kostic  2022  Eintracht Frankfurt    77       0.021
12283  Filip Kostic  2022             Juventus  2618       0.092
13059  Filip Kostic  2023             Juventus  1876       0.034


In [59]:
player_year_agg = players_df.groupby(['id', 'player_name', 'year']).agg(
    total_time=('time', 'sum'),
    total_npxG=('npxG', 'sum'),
    total_goals=('goals', 'sum'),
    clubs=('team_title', lambda x: ' → '.join(x))
).reset_index()

player_year_agg['npxG_per90_combined'] = (player_year_agg['total_npxG'] / player_year_agg['total_time'] * 90).round(3)
player_year_agg['per90_reliable_combined'] = player_year_agg['total_time'] >= 450

print(player_year_agg[player_year_agg['player_name'] == 'Filip Kostic'])

    id   player_name  year  total_time  total_npxG  total_goals  \
47  64  Filip Kostic  2020        2535    4.114966            4   
48  64  Filip Kostic  2021        2527    2.989068            4   
49  64  Filip Kostic  2022        2695    2.692068            3   
50  64  Filip Kostic  2023        1876    0.708112            0   

                             clubs  npxG_per90_combined  \
47             Eintracht Frankfurt                0.146   
48             Eintracht Frankfurt                0.106   
49  Eintracht Frankfurt → Juventus                0.090   
50                        Juventus                0.034   

    per90_reliable_combined  
47                     True  
48                     True  
49                     True  
50                     True  


In [60]:
trajectory_pool_v2 = player_year_agg[player_year_agg['per90_reliable_combined']].sort_values(['id', 'year']).copy()

# re-attach position (take any single row's position per player-year, or most-common if needed — using first for simplicity)
position_lookup = players_df.groupby(['id', 'year'])['primary_position_minutes'].first().reset_index()
trajectory_pool_v2 = trajectory_pool_v2.merge(position_lookup, on=['id', 'year'], how='left')
trajectory_pool_v2 = trajectory_pool_v2[trajectory_pool_v2['primary_position_minutes'] != 'GK']

def classify_trajectory_v2(group):
    group = group.sort_values('year')
    years = group['year'].values
    if len(group) < 3:
        return 'insufficient_data'
    year_gaps = np.diff(years)
    if not (year_gaps == 1).all():
        return 'non_consecutive_seasons'
    diffs = group['npxG_per90_combined'].diff().dropna().values
    last_three = diffs[-3:] if len(diffs) >= 3 else diffs
    if (last_three > 0).all():
        return 'ascending'
    elif (last_three < 0).all():
        return 'declining'
    else:
        return 'established'

trajectories_v2 = trajectory_pool_v2.groupby('id').apply(classify_trajectory_v2, include_groups=False)
print(trajectories_v2.value_counts())

insufficient_data          2007
established                1072
non_consecutive_seasons     271
ascending                   131
declining                   128
Name: count, dtype: int64


In [61]:
kostic_id = players_df[players_df['player_name'] == 'Filip Kostic']['id'].iloc[0]
print(trajectories_v2.get(kostic_id))

declining


In [62]:
clinical_df = pd.read_csv('../data/processed/clinical_games.csv')

print(clinical_df['carried_by_one_player'].dtype)
print(clinical_df['carried_by_one_player'].unique())

str
<ArrowStringArray>
['not_carried_by_one_player', 'carried_by_one_player']
Length: 2, dtype: str


/var/folders/c3/hm5hq4r93ygcpfzfc4gv1mkh0000gn/T/ipykernel_90459/2816802504.py:1: DtypeWarning: Columns (0: derby_name) have mixed types. Specify dtype option on import or set low_memory=False.
  clinical_df = pd.read_csv('../data/processed/clinical_games.csv')


In [63]:
carried_by_team = clinical_df.groupby(['team_name', 'league', 'year']).agg(
    matches=('carried_by_one_player', 'count'),
    carried_matches=('carried_by_one_player', lambda x: (x == 'carried_by_one_player').sum()),
    avg_top_player_share=('top_player_xGChain_share', 'mean')
).reset_index()

carried_by_team['carried_rate'] = (carried_by_team['carried_matches'] / carried_by_team['matches']).round(3)

print(carried_by_team.sort_values('carried_rate', ascending=False).head(10))

         team_name          league  year  matches  carried_matches  \
289           Metz         ligue_1  2023       34                3   
5          Ajaccio         ligue_1  2022       38                3   
453         Verona         serie_a  2023       38                3   
194        Granada         la_liga  2023       38                2   
410         Spezia         serie_a  2021       38                2   
107          Cadiz         la_liga  2020       38                2   
324        Norwich  premier_league  2021       38                2   
296    Montpellier         ligue_1  2021       38                2   
288           Metz         ligue_1  2021       38                2   
36   Athletic Club         la_liga  2023       38                2   

     avg_top_player_share  carried_rate  
289              0.299471         0.088  
5                0.294421         0.079  
453              0.256368         0.079  
194              0.257289         0.053  
410              0.

In [65]:
team_season = pd.read_csv('../data/processed/team_season.csv')

carried_results = carried_by_team.merge(
    team_season[['team_name', 'league', 'year', 'xGD_attack', 'clinical_rate']],
    on=['team_name', 'league', 'year']
)

print(carried_results.corr(numeric_only=True)[['carried_rate']])

                      carried_rate
year                     -0.019292
matches                   0.088391
carried_matches           0.999087
avg_top_player_share      0.590702
carried_rate              1.000000
xGD_attack               -0.033282
clinical_rate            -0.161712


In [66]:
rosters_derby = rosters_df.merge(
    clinical_df[['team_name', 'league', 'year', 'date', 'is_derby']],
    left_on=['team_name', 'league', 'year'],
    right_on=['team_name', 'league', 'year'],
    how='inner'
) if 'team_name' in rosters_df.columns else None

print('team_name' in rosters_df.columns)

False


In [67]:
shots_df = pd.read_csv('../data/processed/shots_enriched.csv')

rosters_df['team_name'] = np.where(
    rosters_df['h_a'] == 'h',
    rosters_df['match_id'].map(shots_df.drop_duplicates('match_id').set_index('match_id')['h_team']),
    rosters_df['match_id'].map(shots_df.drop_duplicates('match_id').set_index('match_id')['a_team'])
)

print(rosters_df['team_name'].isna().sum(), 'unmatched rows')

# join on match_id instead of date, since both should have it cleanly
rosters_derby = rosters_df.merge(
    clinical_df[['team_name', 'league', 'year', 'match_id', 'is_derby']],
    on=['team_name', 'league', 'year', 'match_id'],
    how='inner'
)

print(rosters_derby.shape, rosters_df.shape)

0 unmatched rows
(273825, 26) (273825, 25)


In [68]:
derby_xgchain = rosters_derby.groupby('is_derby')['xGChain'].agg(['mean', 'median', 'count']).round(4)
print(derby_xgchain)

            mean  median   count
is_derby                        
False     0.2728  0.1183  269639
True      0.3206  0.1527    4186


In [69]:
player_derby_counts = rosters_derby.groupby(['player', 'is_derby']).agg(
    appearances=('xGChain', 'count'),
    avg_xGChain=('xGChain', 'mean')
).reset_index()

player_derby_pivot = player_derby_counts.pivot(index='player', columns='is_derby', values=['appearances', 'avg_xGChain'])
player_derby_pivot.columns = ['_'.join(map(str, col)) for col in player_derby_pivot.columns]

# only players with at least 5 derby appearances for a reliable comparison
reliable_derby_players = player_derby_pivot[player_derby_pivot['appearances_True'] >= 5].copy()
reliable_derby_players['derby_diff'] = reliable_derby_players['avg_xGChain_True'] - reliable_derby_players['avg_xGChain_False']

print(reliable_derby_players.sort_values('derby_diff', ascending=False).head(10))
print()
print(reliable_derby_players.sort_values('derby_diff', ascending=True).head(10))

                 appearances_False  appearances_True  avg_xGChain_False  \
player                                                                    
Vitinha                      171.0               7.0           0.501393   
John Stones                   80.0               6.0           0.404614   
Rodri                        133.0               7.0           0.737744   
Raphinha                     159.0              12.0           0.641795   
Marco Verratti                68.0               6.0           0.704924   
Pau Cubarsí                   48.0               6.0           0.446519   
Timothy Weah                 140.0               6.0           0.256532   
Phil Foden                   142.0               9.0           0.609575   
Jack Grealish                114.0               6.0           0.594513   
Andrea Cambiaso              117.0               8.0           0.259164   

                 avg_xGChain_True  derby_diff  
player                                         
Vit

In [70]:
team_season_ha = pd.read_csv('../data/processed/team_season_ha.csv')
print(team_season_ha.columns.tolist())
print(team_season_ha.head())

['team_name', 'league', 'year', 'h_a', 'games_played', 'scored', 'missed', 'xG', 'xGA', 'match_xGD', 'match_npxGD', 'dominance', 'xGD_magnitude', 'dominance_gap', 'gk_save_ratio', 'goals_vs_xG_ratio_team', 'clinical', 'ultra_clinical', 'wasteful', 'ultra_wasteful', 'heist', 'grand_heist', 'robbery', 'grand_robbery', 'dominant_win', 'dropped_points', 'stolen_point', 'heroic_defence', 'gk_worldie', 'gk_nightmare', 'home_bottled', 'away_heist', 'fortress', 'perfect_heist', 'cruel', 'clinical_rate', 'heist_rate', 'robbery_rate', 'wasteful_rate', 'xG_per_game', 'xGA_per_game', 'goals_per_game', 'conceded_per_game', 'xGD_per_game']
  team_name   league  year h_a  games_played  scored  missed         xG  \
0  AC Milan  serie_a  2020   a            19      43      17  37.119001   
1  AC Milan  serie_a  2020   h            19      31      24  37.928325   
2  AC Milan  serie_a  2021   a            19      41      19  34.901264   
3  AC Milan  serie_a  2021   h            19      28      12  32.4

In [71]:
ha_pivot = team_season_ha.pivot_table(
    index=['team_name', 'league', 'year'],
    columns='h_a',
    values=['xGD_per_game', 'clinical_rate', 'heist_rate']
)

ha_pivot.columns = ['_'.join(col) for col in ha_pivot.columns]
ha_pivot = ha_pivot.reset_index()

ha_pivot['home_advantage_xGD'] = (ha_pivot['xGD_per_game_h'] - ha_pivot['xGD_per_game_a']).round(3)
ha_pivot['home_advantage_clinical'] = (ha_pivot['clinical_rate_h'] - ha_pivot['clinical_rate_a']).round(3)

print(ha_pivot.sort_values('home_advantage_xGD', ascending=False).head(10))
print()
print(ha_pivot.sort_values('home_advantage_xGD', ascending=True).head(10))

             team_name          league  year  clinical_rate_a  \
205              Inter         serie_a  2020            0.474   
274    Manchester City  premier_league  2022            0.368   
174           Freiburg      bundesliga  2024            0.118   
381               Roma         serie_a  2024            0.158   
79   Borussia Dortmund      bundesliga  2022            0.412   
199         Hoffenheim      bundesliga  2020            0.294   
60       Bayern Munich      bundesliga  2020            0.529   
308             Napoli         serie_a  2020            0.526   
399            Sevilla         la_liga  2021            0.368   
273    Manchester City  premier_league  2021            0.421   

     clinical_rate_h  heist_rate_a  heist_rate_h  xGD_per_game_a  \
205            0.737         0.000         0.211          -0.224   
274            0.632         0.053         0.053          -0.218   
174            0.412         0.000         0.118          -0.627   
381         

In [72]:
print(ha_pivot.sort_values('home_advantage_xGD', ascending=False)[['team_name', 'league', 'year', 'xGD_per_game_a', 'xGD_per_game_h', 'home_advantage_xGD']].head(10))
print()
print(ha_pivot.sort_values('home_advantage_xGD', ascending=True)[['team_name', 'league', 'year', 'xGD_per_game_a', 'xGD_per_game_h', 'home_advantage_xGD']].head(10))

             team_name          league  year  xGD_per_game_a  xGD_per_game_h  \
205              Inter         serie_a  2020          -0.224           0.785   
274    Manchester City  premier_league  2022          -0.218           0.728   
174           Freiburg      bundesliga  2024          -0.627           0.300   
381               Roma         serie_a  2024          -0.611           0.292   
79   Borussia Dortmund      bundesliga  2022          -0.221           0.576   
199         Hoffenheim      bundesliga  2020          -0.342           0.441   
60       Bayern Munich      bundesliga  2020           0.300           1.057   
308             Napoli         serie_a  2020          -0.081           0.661   
399            Sevilla         la_liga  2021          -0.203           0.527   
273    Manchester City  premier_league  2021          -0.215           0.510   

     home_advantage_xGD  
205               1.009  
274               0.946  
174               0.927  
381            

In [73]:
print(ha_pivot.sort_values('home_advantage_xGD', ascending=True)[['team_name', 'league', 'year', 'xGD_per_game_a', 'xGD_per_game_h', 'home_advantage_xGD']].head(10).to_string())

         team_name          league  year  xGD_per_game_a  xGD_per_game_h  home_advantage_xGD
469  Werder Bremen      bundesliga  2024           0.416          -0.549              -0.965
311         Napoli         serie_a  2023           0.125          -0.835              -0.960
97           Brest         ligue_1  2023           0.418          -0.356              -0.774
298    Montpellier         ligue_1  2023           0.162          -0.611              -0.773
267       Mainz 05      bundesliga  2024           0.355          -0.403              -0.758
118     Celta Vigo         la_liga  2023           0.101          -0.648              -0.749
331        Osasuna         la_liga  2023           0.321          -0.362              -0.683
164  FC Heidenheim      bundesliga  2024           0.165          -0.516              -0.681
228          Leeds  premier_league  2020           0.412          -0.268              -0.680
360  Real Sociedad         la_liga  2021          -0.084          -0.7

In [74]:
print(team_season_ha[(team_season_ha['team_name'] == 'Napoli') & (team_season_ha['year'] == 2023)][['h_a', 'scored', 'missed', 'xG', 'xGA', 'clinical_rate', 'heist_rate']])

    h_a  scored  missed         xG        xGA  clinical_rate  heist_rate
622   a      31      21  28.625351  17.011639          0.421       0.000
623   h      24      27  39.864044  24.878254          0.211       0.053


In [77]:
def build_seasonal_rag(league):
    league_row = league_full[league_full['league'] == league].sort_values('year')
    
    league_2023_z = shots_anomalies[(shots_anomalies['league'] == league)].sort_values('avg_shots_per_match_zscore', ascending=False).iloc[0]
    
    top_shot_team = team_season_shots[team_season_shots['league'] == league].sort_values('total_shots_zscore', ascending=False).iloc[0]
    top_conv_team = team_season_shots[team_season_shots['league'] == league].sort_values('conversion_zscore', ascending=False).iloc[0]
    
    ha_league = ha_pivot[ha_pivot['league'] == league].sort_values('home_advantage_xGD', ascending=False)
    top_home_team = ha_league.iloc[0]
    top_away_team = ha_league.iloc[-1]
    
    block = f"""
LEAGUE: {league.replace('_', ' ').title()}

SEASONAL TRENDS (2020-2024):
avg_xG_per_match_2020={league_row.iloc[0]['avg_xG_per_match']:.3f} | avg_xG_per_match_2024={league_row.iloc[-1]['avg_xG_per_match']:.3f}
NOTE: 2020 seasons show a league-wide dip across most leagues, consistent with COVID-era disruption (empty stadiums, congested fixtures)
2023_ANOMALY: {league} shot volume in 2023 shows z-score={league_2023_z['avg_shots_per_match_zscore']:.2f} vs own 5-season history — part of a multi-league shot-volume spike that year, confirmed real (not a data completeness issue)

TEAM-LEVEL ANOMALIES:
highest_shot_volume_season: {top_shot_team['team_name']} {int(top_shot_team['year'])} (z={top_shot_team['total_shots_zscore']:.2f})
highest_conversion_season: {top_conv_team['team_name']} {int(top_conv_team['year'])} (z={top_conv_team['conversion_zscore']:.2f})

PLAYER ANOMALY LAYER ("MESSI REEL"):
z-score anomaly detection uses minutes-weighted primary position (validated against string-based heuristic — heuristic agreed with ground truth only 62.4% of the time, minutes-weighted version used)
GK excluded from npxG-based anomaly checks — near-zero variance in that group produces unstable, meaningless z-scores
Attacking full-backs/wing-backs (e.g. Robin Gosens 2020, Jeremie Frimpong 2023) surface as legitimate defensive-position outliers on attacking metrics — reflects real modern tactical trend, not a classification error

PLAYER PEAK-SEASON DETECTION:
compares each player to their OWN career mean (not league peers) — 197 peak seasons flagged (npxG_per90 > personal mean + 1.5 std)
notable validated peaks: Marco Asensio 2024 (PSG), Rafael Leão 2022 (AC Milan)

PENALTY DEPENDENCY:
xG - npxG per player, filtered to xG >= 3.0 to exclude single-penalty-fluke seasons
top penalty-reliant players (multi-season validated): Jorginho (Chelsea, 2020 & 2021, 90%+ dependency), Domenico Criscito (Genoa 2021), Thomas Mangani (Angers, 2020 & 2021)

SHOT LOCATION EVOLUTION (global, all leagues):
forwards' six_yard_box shot share rose from 71.1% (2020) to 74.0% (2024) — shots trending closer to goal, not further as originally hypothesized
midfielders show the largest shift: six_yard_box share 47.0% (2020) to 53.1% (2024) — reflects more penalty-box runs by midfielders over time
left/right shot split stable at ~48/52 across all positions and seasons — small persistent right-side lean, no meaningful trend
extreme wide-angle shots (Y<0.2 or Y>0.8) are universally rare (<1.5% of shots) at every position and team

CONVERSION BY SHOT LOCATION (global):
centre shots convert at 15.13% vs 5.6-5.8% for mid-flank zones — roughly 2.5-3x premium for central positions
forwards convert centre chances at 20.0%, nearly double defenders' centre conversion (10.3%) — a genuine finishing-skill signal, not just position

CAREER TRAJECTORY (season-over-season, NOT age-based — no birthdate data available):
requires 3+ consecutive reliable seasons; mid-season transfers aggregated via minutes-weighted combination (preserving club history in a separate field)
{trajectories_v2.value_counts().get('ascending', 0)} players ascending, {trajectories_v2.value_counts().get('declining', 0)} declining, {trajectories_v2.value_counts().get('established', 0)} established (global counts)

CARRIED-BY-ONE-PLAYER:
rare event even for highest-rate teams (max 8.8% of matches in a season) — does not correlate meaningfully with xGD_attack (r=-0.033) or clinical_rate (r=-0.162)

DERBY PERFORMANCE (PLAYER LEVEL):
players show notably higher xGChain in derbies (mean 0.321) vs regular matches (mean 0.273)
biggest derby performers (5+ derby appearances): Vitinha, Rodri, Raphinha, John Stones — matches known "big game player" reputations

HOME ADVANTAGE DECOMPOSITION (this league):
strongest home fortress: {top_home_team['team_name']} {int(top_home_team['year'])} (home_advantage_xGD={top_home_team['home_advantage_xGD']:.3f})
most reversed pattern (better away than home): {top_away_team['team_name']} {int(top_away_team['year'])} (home_advantage_xGD={top_away_team['home_advantage_xGD']:.3f})
NOTABLE CASE: Napoli 2023 (Serie A title-winning season) showed a home finishing collapse — home clinical_rate 0.211 vs away 0.421, despite creating MORE xG at home (39.86) than away (28.63). Higher home xG did not translate to goals; a genuine "home nerves" pattern distinct from simple away-preference.

INSIGHT: {league.replace('_',' ').title()}'s seasonal data reveals football trending toward higher-quality, closer-range shot selection over 2020-2024, with midfielders increasingly making penalty-box runs. Player-level analysis surfaces both consistent attacking identities (multi-season penalty specialists, big-game performers) and a reminder that raw home-advantage assumptions can mask more specific patterns like Napoli's home finishing struggles during a title-winning campaign.
"""
    return block.strip()

for league in league_full['league'].unique():
    text = build_seasonal_rag(league)
    with open(f'../data/rag_findings/seasonal_trends_{league}.txt', 'w') as f:
        f.write(text)
    print(f"saved {league}")

print(build_seasonal_rag('serie_a'))

saved bundesliga
saved la_liga
saved ligue_1
saved premier_league
saved serie_a
LEAGUE: Serie A

SEASONAL TRENDS (2020-2024):
avg_xG_per_match_2020=2.995 | avg_xG_per_match_2024=2.723
NOTE: 2020 seasons show a league-wide dip across most leagues, consistent with COVID-era disruption (empty stadiums, congested fixtures)
2023_ANOMALY: serie_a shot volume in 2023 shows z-score=1.22 vs own 5-season history — part of a multi-league shot-volume spike that year, confirmed real (not a data completeness issue)

TEAM-LEVEL ANOMALIES:
highest_shot_volume_season: Napoli 2023 (z=2.50)
highest_conversion_season: Lazio 2021 (z=2.62)

PLAYER ANOMALY LAYER ("MESSI REEL"):
z-score anomaly detection uses minutes-weighted primary position (validated against string-based heuristic — heuristic agreed with ground truth only 62.4% of the time, minutes-weighted version used)
GK excluded from npxG-based anomaly checks — near-zero variance in that group produces unstable, meaningless z-scores
Attacking full-back

In [79]:
import json

with open('../eval/eval_questions.json', 'r') as f:
    eval_questions = json.load(f)

print(f"Current total: {len(eval_questions)}")

Current total: 26


In [80]:
with open('../eval/eval_questions.json', 'r') as f:
    eval_questions = json.load(f)

print(f"Current total: {len(eval_questions)}")

new_questions = [
    {
        "question": "Did Napoli perform better at home or away during their 2023 title-winning season?",
        "tool": "RAG",
        "answer": "Surprisingly, away — Napoli's home clinical_rate was 0.211 vs 0.421 away, despite creating more xG at home (39.86) than away (28.63). A home finishing collapse, not a preference for away venues."
    },
    {
        "question": "Are strikers shooting from further out in recent seasons?",
        "tool": "RAG",
        "answer": "No — the opposite. Forwards' six-yard-box shot share rose from 71.1% (2020) to 74.0% (2024), meaning shots are trending closer to goal, not further away."
    },
    {
        "question": "Which players show the biggest jump in attacking involvement during derby matches?",
        "tool": "RAG",
        "answer": "Vitinha, Rodri, Raphinha, and John Stones show the largest xGChain increases in derbies vs regular matches, consistent with reputations as big-game performers"
    },
    {
        "question": "Who was the most penalty-reliant player across multiple seasons?",
        "tool": "RAG",
        "answer": "Jorginho at Chelsea — over 90% of his expected goals came from penalties in both 2020 and 2021, confirmed as a sustained pattern, not a single-season fluke"
    },
    {
        "question": "Does a team getting carried by one standout player correlate with better results?",
        "tool": "RAG",
        "answer": "No — carried-by-one-player is rare (max 8.8% of matches even for the most reliant teams) and shows no meaningful correlation with attacking output (r=-0.033) or clinical finishing (r=-0.162)"
    },
    {
        "question": "Was there a real, verified shift in shot volume across Europe in 2023?",
        "tool": "RAG",
        "answer": "Yes — Bundesliga, Premier League, and Ligue 1 all showed their statistically highest shot-volume season in 2023, confirmed as real signal (not a data completeness issue) via cross-checking match counts"
    }
]

eval_questions.extend(new_questions)

with open('../eval/eval_questions.json', 'w') as f:
    json.dump(eval_questions, f, indent=2)

print(f"Total questions now: {len(eval_questions)}")

Current total: 26
Total questions now: 32
